# PILOT behavior cloning v2q: Qwen2.5-7B-Instruct LoRA on quality-selected Gym data

Same data, LoRA rank, learning rate, epochs, and effective batch as the Deep Research Bench
notebook (`notebooks/pilot_bc_qwen_lora.ipynb`). Loss is only on the assistant turn
(KEEP / ALLOC / DECISION). The Bench run quantized the base to 4-bit so it would fit
on a T4. This one keeps Qwen2.5-7B-Instruct in bf16, which is what an A100 is for.
It does not touch the Bench adapter.

Data is the Gym v2q export: the top-2 rollouts per query **by report quality** from the
300-rollout Gym corpus (v1 cloned the top decile by R(tau), which mostly tracked final
coverage and cost). 136 train rows, 9 dev rows (dev queries 1004217, 969384, as in v1):

- `data/bc/gym_v2q/oai_train.jsonl`
- `data/bc/gym_v2q/oai_dev.jsonl`

Built with `adr bc-pairs runs/gym_corpus_v1/rollouts_gym --select-by quality --top-frac 0.01 --min-per-query 2 --max-per-query 2 --reward-config configs/reward_gym.yaml`.

The adapter is saved as `qwen25-7b-pilot-bc-gym-v2q` under `MyDrive/pilot_bc/adapters/`
and served as model `pilot-bc-v2q`. The v1 adapter is not touched.

**Runtime:** Runtime → Change runtime type → A100. The train cell stops if the GPU is not an A100.
Serving uses vLLM, which needs this GPU.

**Before the data cell:** copy those two jsonl files to
`MyDrive/pilot_bc/data/gym_v2q/`, or upload them when that cell asks.

**Sections**
1. Setup + data
2. Train (bf16 LoRA)
3. Dev check (format validity, KEEP Jaccard, DECISION accuracy)
4. Save adapter to Drive
5. Serve with vLLM + tunnel → copy the URL to the laptop

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q -U "transformers>=4.46" "peft>=0.13" "trl>=0.21" "accelerate>=1.0" "datasets>=3.0"
# Colab ships torchao 0.10. Current peft raises if that package is present and older than 0.16,
# even though this run never uses it. An absent torchao is accepted.
%pip uninstall -y -q torchao
import torch, transformers, trl, peft
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
print("gpu:", torch.cuda.get_device_name(0), f"{props.total_memory/1024**3:.0f} GiB", "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Adapter: MyDrive/pilot_bc/adapters/qwen25-7b-pilot-bc-gym-v2q
# Data:    MyDrive/pilot_bc/data/gym_v2q/oai_{train,dev}.jsonl
#          laptop copies: data/bc/gym_v2q/oai_train.jsonl (136 rows), oai_dev.jsonl (9)
CORPUS = "gym_v2q"
RUN_NAME = "qwen25-7b-pilot-bc-gym-v2q"
N_TRAIN, N_DEV = 136, 9

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pilot_bc")
DATA = ROOT / "data" / CORPUS
ADAPTERS = ROOT / "adapters"
DATA.mkdir(parents=True, exist_ok=True)
ADAPTERS.mkdir(parents=True, exist_ok=True)

TRAIN = DATA / "oai_train.jsonl"
DEV = DATA / "oai_dev.jsonl"

def n_lines(path):
    if not path.is_file():
        return 0
    return sum(1 for line in path.read_text().splitlines() if line.strip())

# 104/9 is the Bench export, 92/8 the Gym v1 export. Gym v2q is 136/9.
if n_lines(TRAIN) != N_TRAIN or n_lines(DEV) != N_DEV:
    print(
        f"Found train={n_lines(TRAIN)} dev={n_lines(DEV)} in {DATA}.\n"
        "Upload the Gym v2q files from the laptop: data/bc/gym_v2q/oai_train.jsonl and oai_dev.jsonl.\n"
        "Not data/bc/gym/ (Gym v1, 92 rows) or data/bc/oai_train.jsonl (Bench, 104 rows)."
    )
    from google.colab import files
    up = files.upload()
    for name, blob in up.items():
        # Colab renames a re-upload to "oai_train (2).jsonl" when the name is taken;
        # save it under the name the check reads.
        import re
        clean = re.sub(r" \(\d+\)(?=\.jsonl$)", "", Path(name).name)
        (DATA / clean).write_bytes(blob)
if n_lines(TRAIN) != N_TRAIN or n_lines(DEV) != N_DEV:
    raise RuntimeError(
        f"Still train={n_lines(TRAIN)} dev={n_lines(DEV)}. "
        "Replace MyDrive/pilot_bc/data/gym_v2q/ with data/bc/gym_v2q/oai_train.jsonl (136) and oai_dev.jsonl (9)."
    )
print("corpus:", CORPUS, "| run:", RUN_NAME)
print("train:", TRAIN, n_lines(TRAIN), "| dev:", DEV, n_lines(DEV))

In [ ]:
import json
from datasets import Dataset

def load_pc(path):
    # OpenAI chat rows -> TRL conversational prompt/completion (loss on the assistant turn only)
    rows = []
    for line in open(path):
        m = json.loads(line)["messages"]
        assert m[-1]["role"] == "assistant"
        rows.append({"prompt": m[:-1], "completion": [m[-1]]})
    return Dataset.from_list(rows)

train_ds = load_pc(TRAIN)
dev_ds = load_pc(DEV)
if len(train_ds) != N_TRAIN or len(dev_ds) != N_DEV:
    raise RuntimeError(
        f"Loaded train={len(train_ds)} dev={len(dev_ds)} from {TRAIN}. "
        "Expected the Gym v2q export. Re-run the data cell and upload data/bc/gym_v2q/oai_*.jsonl."
    )
print(train_ds, dev_ds)
print(train_ds[0]["completion"][0]["content"])

## 2. Train (bf16 base, LoRA r=16 on all projections)

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-7B-Instruct"
# RUN_NAME is set in the data cell (qwen25-7b-pilot-bc-gym-v2q).
OUT = f"/content/out/{RUN_NAME}"
gpu_name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"gpu {gpu_name} {mem_gb:.0f} GiB")
if "A100" not in gpu_name:
    raise RuntimeError(
        f"This notebook trains in bf16 for an A100, and the runtime is {gpu_name}. "
        "Runtime → Change runtime type → A100."
    )
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("A100 should support bf16")
DTYPE = torch.bfloat16
# Effective batch stays 8, matching the Bench run. A 40 GB A100 cannot hold a
# microbatch of 2 at 8192 tokens in bf16; 80 GB can.
BATCH, ACCUM = (2, 4) if mem_gb >= 70 else (1, 8)
print(f"microbatch {BATCH} x accum {ACCUM}")

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token

def n_tokens(ex):
    out = tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=True, add_generation_prompt=False)
    ids = out["input_ids"] if not isinstance(out, list) else out
    if ids and isinstance(ids[0], (list, tuple)):
        ids = ids[0]
    return len(ids)
# Truncation would cut the completion, which is the label, so over-long rows are dropped.
CAP = 8192
train_ds = train_ds.filter(lambda ex: n_tokens(ex) <= CAP)
dev_ds = dev_ds.filter(lambda ex: n_tokens(ex) <= CAP)
lens = [n_tokens(ex) for ex in train_ds]
print(f"kept train {len(train_ds)} dev {len(dev_ds)} | tokens mean {sum(lens)/len(lens):.0f} max {max(lens)}")
MAX_LEN = CAP

# A re-run of this cell must drop the previous model first. Otherwise the A100
# still holds ~15 GB of weights and from_pretrained tries to load a second copy.
import gc
for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=DTYPE, device_map={"": 0}, attn_implementation="sdpa")
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=ACCUM,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,            # same schedule as the Bench run (~36 steps here)
    weight_decay=0.0,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LEN,
    packing=False,
    completion_only_loss=True,   # loss on KEEP/ALLOC/DECISION only
    report_to="none",
    seed=17,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=dev_ds,
                     processing_class=tok, peft_config=lora)
trainer.train()
print(trainer.state.log_history[-3:])

## 3. Dev check: does it emit valid actions, and how close to the cloned ones?

In [ ]:
import re, torch

_KEEP = re.compile(r"^\s*KEEP\s*:\s*(.*)$", re.I | re.M)
_ALLOC = re.compile(r"^\s*ALLOC\s*:\s*(.*)$", re.I | re.M)
_DEC = re.compile(r"^\s*DECISION\s*:\s*(CONTINUE|TERMINATE)", re.I | re.M)
_PAIR = re.compile(r"([A-Za-z0-9_\-]+)\s*=\s*([0-9]*\.?[0-9]+)")

def parse(text):
    k = _KEEP.search(text or ""); a = _ALLOC.search(text or ""); d = _DEC.search(text or "")
    keep = None
    if k:
        body = k.group(1).strip()
        keep = "ALL" if body.upper() == "ALL" else set(re.findall(r"[A-Za-z0-9_\-]+", body))
    alloc = {n: float(w) for n, w in _PAIR.findall(a.group(1))} if a else {}
    s = sum(alloc.values()) or 1.0
    alloc = {n: w / s for n, w in alloc.items()}
    dec = d.group(1).upper() if d else None
    return keep, alloc, dec

def pool_ids(prompt_text):
    ids = []
    grab = False
    for line in prompt_text.splitlines():
        if line.startswith("Evidence ("): grab = True; continue
        if grab:
            if not line.strip(): break
            ids.append(line.strip().split(" | ")[0])
    return ids

def encode_chat(messages):
    # Newer tokenizers return a BatchEncoding dict; older ones a bare tensor.
    out = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    ids = out["input_ids"] if not isinstance(out, torch.Tensor) else out
    return ids.to(model.device)

@torch.no_grad()
def generate(messages, max_new_tokens=1200):
    # Each 12-char evidence id is ~9 Qwen tokens; gold completions run up to ~930.
    ids = encode_chat(messages)
    out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

model.eval()
model.config.use_cache = True
rows = []
for ex in dev_ds:
    pred = generate(ex["prompt"])
    gold = ex["completion"][0]["content"]
    pk, pa, pd_ = parse(pred); gk, ga, gd = parse(gold)
    pool = set(pool_ids(ex["prompt"][-1]["content"]))
    pk = pool if pk == "ALL" else (pk or set()); gk = pool if gk == "ALL" else (gk or set())
    jacc = len(pk & gk) / len(pk | gk) if (pk | gk) else 1.0
    l1 = sum(abs(pa.get(n, 0) - ga.get(n, 0)) for n in set(pa) | set(ga))
    rows.append(dict(valid=bool(pk) and pd_ is not None, keep_jaccard=round(jacc, 3), keep_frac_pred=round(len(pk)/max(1,len(pool)),2),
                     keep_frac_gold=round(len(gk)/max(1,len(pool)),2), alloc_l1=round(l1, 3), dec_pred=pd_, dec_gold=gd, dec_ok=pd_ == gd,
                     hallucinated_ids=len(pk - pool)))
import pandas as pd
df = pd.DataFrame(rows); display(df)
print("valid %.2f | KEEP Jaccard %.3f | DECISION acc %.2f | ALLOC L1 %.3f | hallucinated ids %d" %
      (df.valid.mean(), df.keep_jaccard.mean(), df.dec_ok.mean(), df.alloc_l1.mean(), df.hallucinated_ids.sum()))
print("\n--- sample ---\n", pred)

## 4. Save the adapter to Drive

In [ ]:
SAVE = ADAPTERS / RUN_NAME
trainer.model.save_pretrained(str(SAVE))
tok.save_pretrained(str(SAVE))
(SAVE / "train_log.json").write_text(json.dumps(trainer.state.log_history, indent=1))
print("saved:", sorted(p.name for p in SAVE.iterdir()))

## 5. Serve it for the laptop

On the A100, serve the saved adapter with vLLM. Skip the FastAPI cell: that server
keeps the training model on the GPU, and vLLM needs the card.

The vLLM cell reads `MyDrive/pilot_bc/adapters/qwen25-7b-pilot-bc-gym-v2q`. After a
runtime restart, run that cell directly. The reload cell is only for re-running the
dev check without training again.

The last cell prints the `GR_ORCH_LLM_*` exports to paste on the laptop. Keep the
notebook running while the eval runs; the URL dies with the runtime.

In [ ]:
# Optional. Only for re-running the dev check after a restart. Skip this when serving:
# vLLM loads the adapter itself, and this copy would sit on the A100 next to it.
if "model" not in globals():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    from pathlib import Path
    BASE = "Qwen/Qwen2.5-7B-Instruct"
    RUN_NAME = "qwen25-7b-pilot-bc-gym-v2q"
    ADAPTERS = Path("/content/drive/MyDrive/pilot_bc/adapters"); SAVE = ADAPTERS / RUN_NAME
    tok = AutoTokenizer.from_pretrained(str(SAVE))
    base = AutoModelForCausalLM.from_pretrained(
        BASE, torch_dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
    model = PeftModel.from_pretrained(base, str(SAVE)).eval()
    print("reloaded", SAVE)

In [ ]:
# Skip on the A100. The next cell serves the adapter with vLLM.
print("Run the vLLM cell. Do not start a second server on port 8000.")

In [ ]:
# Serve the Gym v2q adapter as model name "pilot-bc-v2q" (set GR_ORCH_LLM_MODEL to it on the laptop).
%pip install -q vllm

import gc, json, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

BASE = "Qwen/Qwen2.5-7B-Instruct"
SAVE = Path("/content/drive/MyDrive/pilot_bc/adapters/qwen25-7b-pilot-bc-gym-v2q")
assert (SAVE / "adapter_config.json").is_file(), f"missing adapter at {SAVE}; run the save cell first"

# vLLM loads its own weights. Free the training model so a 40 GB A100 has the card.
for _name in ("trainer", "model", "base"):
    if _name in globals():
        del globals()[_name]
gc.collect()
import torch
torch.cuda.empty_cache()

# A vLLM left over from an earlier run of this cell holds ~15 GB and makes the new
# one fail at startup. Stop it, then ask vLLM only for memory that is actually free.
subprocess.run(["pkill", "-f", "vllm serve"], check=False)
time.sleep(5)
free_b, total_b = torch.cuda.mem_get_info()
free_gb, total_gb = free_b / 1024**3, total_b / 1024**3
print(f"GPU free {free_gb:.1f} / {total_gb:.1f} GiB")
if free_gb < 22:
    raise RuntimeError(
        f"Only {free_gb:.1f} GiB free; this kernel still holds a model. "
        "Runtime -> Restart session, then run only this cell and the tunnel cell."
    )
GPU_UTIL = round(min(0.90, (free_gb - 1.0) / total_gb), 2)

PORT = 8000
lora_arg = json.dumps({"name": "pilot-bc-v2q", "path": str(SAVE), "base_model_name": BASE})
log = open("/content/vllm.log", "w")
vllm_proc = subprocess.Popen(
    ["vllm", "serve", BASE, "--enable-lora", "--lora-modules", lora_arg,
     "--max-model-len", "16384", "--max-lora-rank", "16",
     "--dtype", "bfloat16", "--port", str(PORT),
     "--gpu-memory-utilization", str(GPU_UTIL)],
    stdout=log, stderr=subprocess.STDOUT,
)
import requests
ready = False
for _ in range(120):
    time.sleep(5)
    if vllm_proc.poll() is not None:
        break
    try:
        requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=5).raise_for_status()
        ready = True
        break
    except Exception:
        pass
print(open("/content/vllm.log").read()[-2000:])
if not ready:
    raise RuntimeError("vLLM did not come up; see the log above")
print(requests.get(f"http://127.0.0.1:{PORT}/v1/models").json())

In [ ]:
# Cloudflare quick tunnel -> public https URL for the laptop
import subprocess, re, time, os
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m: url = m.group(0)
assert url, "tunnel did not come up; re-run this cell"

import socket, requests
host = url.split("/")[2]
print("waiting for DNS", host)
resolved = False
for _ in range(24):
    try:
        socket.getaddrinfo(host, 443)
        resolved = True
        break
    except socket.gaierror:
        time.sleep(5)
print("resolved" if resolved else "Colab DNS still cannot see the hostname; the laptop often can")

if resolved:
    # Short probe only. A full dev prompt is several thousand tokens and hides a DNS failure.
    r = requests.post(
        f"{url}/v1/chat/completions",
        json={"model": "pilot-bc-v2q", "messages": [{"role": "user", "content": "Reply with exactly: DECISION: CONTINUE"}], "max_tokens": 32},
        timeout=180,
    )
    print("status", r.status_code)
    print(r.json()["choices"][0]["message"]["content"][:400])

print("\n=== paste on the laptop (deepresearchagent/) ===")
print(f"export GR_ORCH_LLM_BASE_URL={url}/v1")
print("export GR_ORCH_LLM_MODEL=pilot-bc-v2q")
print("export GR_ORCH_LLM_API_KEY=EMPTY")
print("Keep this notebook running while the eval runs; the URL dies with the runtime.")